# Monthly entry and exit attribution

**Status:** Historical, single-seed matched-pair result; both arms failed the account-survival gate. **Reproduction:** R2 for monthly tables, bootstrap intervals, decomposition statistics, and plots recomputed from the tracked paired-trade rows. **Tier:** A, CPU.

This notebook does not replay checkpoints, read market data, retrain, or evaluate the exposed 2026 interval. It verifies that the tracked per-entry paired-trade CSVs reproduce the reported monthly aggregates and March-to-April decomposition. The underlying run ledgers, ticks, checkpoints, and full environment are not part of this packet, so this is not R3 replay or R4 retraining.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
EVIDENCE = ROOT / "evidence/research/entry_and_exit_attribution"
manifest = json.loads((EVIDENCE / "monthly_entry_exit_summary.json").read_text(encoding="utf-8"))
arm_files = {
    "A4A: hidden entry FSM": (
        "treatment_paired_trades.csv",
        "ladder_a4a_entry_fsm_hidden_qgrid025",
    ),
    "A4B: entry FSM": ("control_paired_trades.csv", "ladder_a4b_entry_fsm_qgrid025"),
}
print("campaign fingerprint:", manifest["campaign_fingerprint"])
print("bootstrap:", manifest["bootstrap"])
print("tracked arms:", list(arm_files))

campaign fingerprint: b37580606ca374326bce7207fb7eef118cede36cf0eaad25d37be9465bb4e1e4
bootstrap: {'resamples': 10000, 'seed': 20261025}
tracked arms: ['A4A: hidden entry FSM', 'A4B: entry FSM']


In [2]:
def bootstrap_mean_ci(values, *, resamples, seed):
    """Reproduce the paired-bootstrap mean interval used by the archived analysis."""
    values = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(resamples, dtype=np.float64)
    for index in range(resamples):
        means[index] = float(np.mean(values[rng.integers(0, values.size, values.size)]))
    return np.quantile(means, [0.025, 0.975])


def recompute_monthly(rows, seed, resamples):
    """Recompute reported monthly means, medians, sums, and confidence intervals."""
    result = []
    for month, group in rows.groupby("entry_month", sort=True):
        row = {"entry_month": str(month), "n": int(len(group))}
        metrics = (
            ("actual_gross_mid_r", "actual_gross_mid"),
            ("actual_historical_net_r", "actual_historical_net"),
            ("ema_gross_r", "ema_reference_gross"),
            ("exit_delta_r", "actual_minus_ema_exit_delta"),
        )
        for column, prefix in metrics:
            values = group[column].dropna().to_numpy(dtype=np.float64)
            row[f"{prefix}_mean_r"] = float(values.mean())
            row[f"{prefix}_median_r"] = float(np.median(values))
            if column in {"actual_gross_mid_r", "actual_historical_net_r"}:
                row[f"{prefix}_ci95_mean_r"] = bootstrap_mean_ci(
                    values, resamples=resamples, seed=seed
                ).tolist()
        row["mid_to_bid_ask_drag_mean_r"] = float(group["actual_fill_cost_drag_r"].mean())
        row["gross_mid_pnl_usd_sum"] = float(group["actual_gross_mid_usd"].sum())
        row["historical_net_pnl_usd_sum"] = float(group["actual_raw_bid_ask_net_usd"].sum())
        result.append(row)
    return result


def assert_close(actual, expected, *, atol=1e-9):
    """Require a recomputed floating-point value to match the recorded result."""
    assert np.isclose(actual, expected, rtol=0.0, atol=atol), (actual, expected)


seed = int(manifest["bootstrap"]["seed"])
resamples = int(manifest["bootstrap"]["resamples"])
recomputed = {}
for label, (filename, manifest_arm_key) in arm_files.items():
    rows = pd.read_csv(EVIDENCE / filename)
    recorded_arm = manifest["arms"][manifest_arm_key]
    assert len(rows) == recorded_arm["replay"]["paired_cohort"]
    assert rows["entry_month"].between("2024-12", "2025-12").all()
    monthly = recompute_monthly(rows, seed, resamples)
    expected = recorded_arm["monthly"]
    assert len(monthly) == len(expected)
    for actual_month, expected_month in zip(monthly, expected, strict=True):
        assert actual_month["entry_month"] == expected_month["entry_month"]
        assert actual_month["n"] == expected_month["n"]
        for key in (
            "actual_gross_mid_mean_r",
            "actual_historical_net_mean_r",
            "ema_reference_gross_mean_r",
            "actual_minus_ema_exit_delta_mean_r",
            "mid_to_bid_ask_drag_mean_r",
            "gross_mid_pnl_usd_sum",
            "historical_net_pnl_usd_sum",
        ):
            assert_close(actual_month[key], expected_month[key])
        for key in (
            "actual_gross_mid_ci95_mean_r",
            "actual_historical_net_ci95_mean_r",
        ):
            np.testing.assert_allclose(actual_month[key], expected_month[key], rtol=0.0, atol=1e-9)
    recomputed[label] = pd.DataFrame(monthly)
    print(label, "rows:", len(rows), "months reproduced:", len(monthly))

print("Monthly counts, means, sums, and 95% bootstrap intervals match the tracked report.")

A4A: hidden entry FSM rows: 987 months reproduced: 13


A4B: entry FSM rows: 825 months reproduced: 11
Monthly counts, means, sums, and 95% bootstrap intervals match the tracked report.


In [3]:
def recompute_march_april(rows, seed, resamples):
    """Recompute the exact paired month-stratified decomposition and interval."""
    march = rows.loc[rows["entry_month"].eq("2025-03")].dropna(
        subset=["actual_gross_mid_r", "ema_gross_r", "exit_delta_r"]
    )
    april = rows.loc[rows["entry_month"].eq("2025-04")].dropna(
        subset=["actual_gross_mid_r", "ema_gross_r", "exit_delta_r"]
    )
    actual_change = float(april.actual_gross_mid_r.mean() - march.actual_gross_mid_r.mean())
    entry_component = float(april.ema_gross_r.mean() - march.ema_gross_r.mean())
    exit_component = float(april.exit_delta_r.mean() - march.exit_delta_r.mean())
    assert np.isclose(actual_change, entry_component + exit_component, atol=1e-12)

    rng = np.random.default_rng(seed)
    bootstrap = np.empty((resamples, 3), dtype=np.float64)
    march_values = [
        march[column].to_numpy(dtype=np.float64)
        for column in ("actual_gross_mid_r", "ema_gross_r", "exit_delta_r")
    ]
    april_values = [
        april[column].to_numpy(dtype=np.float64)
        for column in ("actual_gross_mid_r", "ema_gross_r", "exit_delta_r")
    ]
    for index in range(resamples):
        march_index = rng.integers(0, len(march), len(march))
        april_index = rng.integers(0, len(april), len(april))
        changes = [
            float(
                april_values[column][april_index].mean() - march_values[column][march_index].mean()
            )
            for column in range(3)
        ]
        bootstrap[index] = [changes[0], changes[1], changes[2]]
    intervals = np.quantile(bootstrap, [0.025, 0.975], axis=0).T
    return {
        "paired_entries_march": len(march),
        "paired_entries_april": len(april),
        "actual_change_r": actual_change,
        "entry_cohort_component_r": entry_component,
        "exit_outcome_component_r": exit_component,
        "actual_change_ci95": intervals[0].tolist(),
        "entry_component_ci95": intervals[1].tolist(),
        "exit_component_ci95": intervals[2].tolist(),
    }


for label, (filename, manifest_arm_key) in arm_files.items():
    rows = pd.read_csv(EVIDENCE / filename)
    result = recompute_march_april(rows, seed, resamples)
    recorded = manifest["arms"][manifest_arm_key]["march_april_decomposition"]
    assert result["paired_entries_march"] == recorded["paired_entries_march"]
    assert result["paired_entries_april"] == recorded["paired_entries_april"]
    for result_key, recorded_key in (
        ("actual_change_r", "march_to_april_actual_gross_change_r"),
        ("entry_cohort_component_r", "entry_cohort_component_r"),
        ("exit_outcome_component_r", "exit_outcome_component_r"),
    ):
        assert_close(result[result_key], recorded[recorded_key])
    np.testing.assert_allclose(
        result["actual_change_ci95"],
        recorded["bootstrap_95pct_ci"]["actual_gross_change_r"],
        rtol=0.0,
        atol=1e-9,
    )
    print(label, result)

A4A: hidden entry FSM {'paired_entries_march': 100, 'paired_entries_april': 90, 'actual_change_r': -0.21772192245751507, 'entry_cohort_component_r': -0.12764979748224348, 'exit_outcome_component_r': -0.09007212497527164, 'actual_change_ci95': [-0.5836972820559079, 0.14286408943295933], 'entry_component_ci95': [-0.46095151429056397, 0.22434539708066958], 'exit_component_ci95': [-0.34737623750691343, 0.14062903384611425]}


A4B: entry FSM {'paired_entries_march': 100, 'paired_entries_april': 104, 'actual_change_r': -0.22376042857902093, 'entry_cohort_component_r': -0.1595338908659705, 'exit_outcome_component_r': -0.06422653771305044, 'actual_change_ci95': [-0.5655796837055084, 0.11180292302737999], 'entry_component_ci95': [-0.46536624935409854, 0.15051073664208534], 'exit_component_ci95': [-0.28433107026644383, 0.14565764170351025]}


## Recomputed monthly results

The panels regenerate actual gross mid-quote R and actual historical bid/ask net R from the per-entry CSVs. Their confidence bars use the recorded 10,000-resample seed and the source analysis's paired-bootstrap procedure.

The reproducible figure is saved in the evidence packet as `monthly_entry_exit_attribution.png`.

In [4]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for axis, (label, monthly) in zip(axes, recomputed.items(), strict=True):
    months = monthly["entry_month"]
    for metric, interval, color, name in (
        ("actual_gross_mid", "actual_gross_mid_ci95_mean_r", "#315f8c", "Gross mid-quote R"),
        (
            "actual_historical_net",
            "actual_historical_net_ci95_mean_r",
            "#bd6b3d",
            "Historical bid/ask net R",
        ),
    ):
        means = monthly[f"{metric}_mean_r"].to_numpy()
        bounds = np.asarray(monthly[interval].tolist())
        errors = np.vstack([means - bounds[:, 0], bounds[:, 1] - means])
        axis.errorbar(months, means, yerr=errors, marker="o", capsize=2, color=color, label=name)
    axis.axhline(0.0, color="#666666", linewidth=0.8)
    axis.set_title(label)
    axis.set_ylabel("R per trade")
    axis.legend()
axes[-1].tick_params(axis="x", rotation=45)
axes[-1].set_xlabel("Entry month; training split only")
fig.suptitle("Monthly actual-fill entry and exit outcomes")
fig.tight_layout()
figure_path = EVIDENCE / "monthly_entry_exit_attribution.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.close(fig)
print("Saved figure:", figure_path.relative_to(ROOT))

Saved figure: evidence/research/entry_and_exit_attribution/monthly_entry_exit_attribution.png


## Claim boundary

The rows and report manifest record 987 paired positions for A4A and 825 for A4B, including actual gross mid-quote outcomes, historical net outcomes, and a frozen-entry EMA21 reference. The notebook reproduces their monthly aggregates and the March–April decomposition from those tracked per-entry rows. It does not recreate tick execution or verify the original replay against raw ticks because those ticks, checkpoints, and full run inputs are unavailable in the repository. Both arms failed the declared account-survival gate; this result is not evidence of a viable policy or causal superiority.

See the [semantic evidence packet](https://github.com/nntk072/Quantitative-RL-Trading-System/tree/main/evidence/research/entry_and_exit_attribution), [result registry](result_registry.md), and [active research gate](../operations/research_execution_plan.md). Original input artifacts remain linked by checksum in the evidence manifest.